# ViT5 Fine-tuning for ASR Normalization

This notebook fine-tunes ViT5 to normalize noisy ASR output after KenLM beam search for Vietnamese medical text.
The goal is to correct spelling, diacritics, punctuation, capitalization, spacing, and medical terminology without changing the overall ASR -> KenLM -> ViT5 pipeline.

In [ ]:
%pip install -q --upgrade sentencepiece protobuf evaluate sacrebleu rouge_score

import json
import math
import os
import random
import re
import time
from pathlib import Path
from unicodedata import normalize as unicode_normalize

import evaluate
import numpy as np
import pandas as pd
import torch
from sacrebleu import corpus_bleu
from datasets import Dataset
from huggingface_hub import hf_hub_download, login
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    T5Tokenizer,
    TrainerCallback,
    set_seed,
 )

# Reproducibility
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

def set_global_seed(seed_value: int) -> None:
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
    set_seed(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
    if hasattr(torch, "use_deterministic_algorithms"):
        torch.use_deterministic_algorithms(True, warn_only=True)

set_global_seed(SEED)

# Authentication
hf_token = os.environ.get("HF_TOKEN_READ")
if hf_token:
    login(token=hf_token)
    print("[INFO] Successfully authenticated with HuggingFace Hub using HF_TOKEN_READ.")
else:
    print("[WARNING] HF_TOKEN_READ is not set. Proceeding with anonymous download.")

# Configuration
MODEL_ID = "VietAI/vit5-base"
TRAIN_JSON_PATH = "/kaggle/input/datasets/hdtuznn/dataset-for-finetuned-vit5/finetuned_ViT5/train_vit5.json"
VAL_JSON_PATH = "/kaggle/input/datasets/hdtuznn/dataset-for-finetuned-vit5/finetuned_ViT5/val_vit5.json"
OUTPUT_DIR = Path("/kaggle/working/vit5_medical_rewrite_checkpoints")
FINAL_MODEL_DIR = OUTPUT_DIR / "final_model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 16
MAX_TRAIN_SAMPLES = 25000
MAX_VAL_SAMPLES = 5000
EVAL_STEPS = 1000
SAVE_STEPS = 1000
LOGGING_STEPS = 50
EARLY_STOPPING_PATIENCE = 3

use_cuda = torch.cuda.is_available()
use_bf16 = bool(use_cuda and torch.cuda.is_bf16_supported())
use_fp16 = bool(use_cuda and not use_bf16)
print(f"[INFO] Mixed precision: bf16={use_bf16}, fp16={use_fp16}")

# Model and tokenizer
print("[INFO] Downloading core spiece.model directly to bypass tokenizer corruption...")
spiece_path = hf_hub_download(repo_id=MODEL_ID, filename="spiece.model")
tokenizer = T5Tokenizer(vocab_file=spiece_path, legacy=True)

print(f"[INFO] Downloading base weights: {MODEL_ID}")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
model.config.use_cache = False
model.gradient_checkpointing_enable()

if tokenizer.pad_token_id is not None:
    model.config.decoder_start_token_id = tokenizer.pad_token_id
    model.config.pad_token_id = tokenizer.pad_token_id
    if hasattr(model, "generation_config") and model.generation_config is not None:
        model.generation_config.decoder_start_token_id = tokenizer.pad_token_id
        model.generation_config.pad_token_id = tokenizer.pad_token_id

if tokenizer.eos_token_id is not None:
    model.config.eos_token_id = tokenizer.eos_token_id
    if hasattr(model, "generation_config") and model.generation_config is not None:
        model.generation_config.eos_token_id = tokenizer.eos_token_id

rouge_metric = evaluate.load("rouge")

In [ ]:
# Dataset loading
print(f"[INFO] Loading training data from: {TRAIN_JSON_PATH}")
if os.path.exists(TRAIN_JSON_PATH):
    df_train = pd.read_json(TRAIN_JSON_PATH)
    df_val = pd.read_json(VAL_JSON_PATH)
else:
    df_train = pd.read_json("/kaggle/working/train_vit5.json")
    df_val = pd.read_json("/kaggle/working/val_vit5.json")

required_columns = {"input_text", "target_text"}
if not required_columns.issubset(df_train.columns):
    raise ValueError(f"Training data must contain columns: {sorted(required_columns)}")
if not required_columns.issubset(df_val.columns):
    raise ValueError(f"Validation data must contain columns: {sorted(required_columns)}")

print(f"[INFO] Total train records available: {len(df_train):,}")
print(f"[INFO] Total validation records available: {len(df_val):,}")

if len(df_train) > MAX_TRAIN_SAMPLES:
    df_train = df_train.sample(n=MAX_TRAIN_SAMPLES, random_state=SEED).reset_index(drop=True)
else:
    df_train = df_train.reset_index(drop=True)

if len(df_val) > MAX_VAL_SAMPLES:
    df_val = df_val.sample(n=MAX_VAL_SAMPLES, random_state=SEED).reset_index(drop=True)
else:
    df_val = df_val.reset_index(drop=True)

raw_train_dataset = Dataset.from_pandas(df_train, preserve_index=False)
raw_val_dataset = Dataset.from_pandas(df_val, preserve_index=False)
print(f"[INFO] Training samples used: {len(raw_train_dataset):,}")
print(f"[INFO] Validation samples used: {len(raw_val_dataset):,}")

In [ ]:
# Preprocessing and metrics
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def normalize_text(text: str) -> str:
    text = unicode_normalize("NFC", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_predictions = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    decoded_predictions = [normalize_text(text) for text in decoded_predictions]
    decoded_labels = [normalize_text(text) for text in decoded_labels]

    rouge_scores = rouge_metric.compute(
        predictions=decoded_predictions,
        references=decoded_labels,
        use_stemmer=False,
    )
    bleu_score = corpus_bleu(decoded_predictions, [decoded_labels]).score
    exact_match = float(np.mean([pred == ref for pred, ref in zip(decoded_predictions, decoded_labels)]))

    return {
        "rougeL": round(rouge_scores["rougeL"] * 100.0, 4),
        "bleu": round(bleu_score, 4),
        "exact_match": round(exact_match * 100.0, 4),
    }

def preview_predictions(trainer, raw_dataset, tokenized_dataset, num_examples=3):
    sample_size = min(num_examples, len(raw_dataset))
    sample_indices = np.linspace(0, len(raw_dataset) - 1, num=sample_size, dtype=int).tolist()
    raw_samples = raw_dataset.select(sample_indices)
    tokenized_samples = tokenized_dataset.select(sample_indices)
    prediction_output = trainer.predict(
        tokenized_samples,
        metric_key_prefix="preview",
        max_length=MAX_TARGET_LENGTH,
        num_beams=4,
    )
    prediction_ids = prediction_output.predictions
    if isinstance(prediction_ids, tuple):
        prediction_ids = prediction_ids[0]

    decoded_predictions = tokenizer.batch_decode(
        prediction_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    decoded_predictions = [normalize_text(text) for text in decoded_predictions]

    examples = []
    print("\n[INFO] Qualitative prediction examples:")
    for raw_row, prediction in zip(raw_samples, decoded_predictions):
        example = {
            "input_text": raw_row["input_text"],
            "target_text": raw_row["target_text"],
            "prediction": prediction,
        }
        examples.append(example)
        print("-" * 80)
        print(f"INPUT     : {example['input_text']}")
        print(f"REFERENCE : {example['target_text']}")
        print(f"PREDICT   : {example['prediction']}")

    return examples

tokenized_train = raw_train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_train_dataset.column_names,
    desc="Tokenizing Train Data",
 )
tokenized_val = raw_val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_val_dataset.column_names,
    desc="Tokenizing Val Data",
 )

In [ ]:
# Training callbacks and arguments
class DetailedProgressCallback(TrainerCallback):
    def __init__(self):
        super().__init__()
        self.start_time = time.time()

    def on_epoch_begin(self, args, state, control, **kwargs):
        current_epoch = 1 if state.epoch is None else int(state.epoch) + 1
        print(f"\n[INFO] ---> STARTING EPOCH {current_epoch} / {int(args.num_train_epochs)} <---")

    def on_epoch_end(self, args, state, control, **kwargs):
        elapsed_min = (time.time() - self.start_time) / 60.0
        epochs_done = max(1, int(math.ceil(state.epoch or 0)))
        total_epochs = int(args.num_train_epochs)
        est_total_min = (elapsed_min / epochs_done) * total_epochs
        rem_min = max(0.0, est_total_min - elapsed_min)
        print(
            f"[INFO] <--- FINISHED EPOCH {epochs_done}/{total_epochs} | "
            f"Elapsed: {elapsed_min:.1f} mins | Remaining ETA: ~{rem_min:.1f} mins --->"
        )

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    learning_rate=3e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    fp16=use_fp16,
    bf16=use_bf16,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    logging_steps=LOGGING_STEPS,
    logging_strategy="steps",
    logging_first_step=True,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    disable_tqdm=False,
 )

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    pad_to_multiple_of=8 if use_cuda else None,
 )

callbacks = [
    DetailedProgressCallback(),
    EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
 ]

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=callbacks,
 )

In [ ]:
# Training, evaluation, and saving
print("[INFO] Starting Seq2Seq fine-tuning for ViT5 ASR normalization...")
train_result = trainer.train()
train_metrics = dict(train_result.metrics)
train_metrics["train_samples"] = len(tokenized_train)
trainer.log_metrics("train", train_metrics)
trainer.save_metrics("train", train_metrics)
trainer.save_state()

eval_metrics = trainer.evaluate(metric_key_prefix="eval")
eval_metrics["eval_samples"] = len(tokenized_val)
trainer.log_metrics("eval", eval_metrics)
trainer.save_metrics("eval", eval_metrics)

preview_examples = preview_predictions(trainer, raw_val_dataset, tokenized_val, num_examples=3)
(FINAL_MODEL_DIR / "preview_examples.json").write_text(
    json.dumps(preview_examples, ensure_ascii=False, indent=2),
    encoding="utf-8",
 )

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
(OUTPUT_DIR / "training_args.json").write_text(
    training_args.to_json_string(),
    encoding="utf-8",
 )

print(f"[SUCCESS] Fine-tuning completed and best model saved to: {FINAL_MODEL_DIR}")
print(f"[SUCCESS] Training arguments saved to: {OUTPUT_DIR / 'training_args.json'}")
print(f"[SUCCESS] Metrics saved under: {OUTPUT_DIR}")

print("\n" + "=" * 80)
print("[INFO] SANITY CHECK: TESTING FINE-TUNED VIT5 MODEL ON A SIMULATED NOISY QUESTION:")
test_noisy_q = "fix_asr: khi nào thì nên uống l cystin mỗi ngày có bị buồn lôn hay đau dạ dày không"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
inputs = tokenizer(test_noisy_q, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_length=MAX_TARGET_LENGTH, num_beams=4)
pred_clean_q = tokenizer.decode(outputs[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)
pred_clean_q = normalize_text(pred_clean_q)

print(f"Input  (Noisy ASR)    : {test_noisy_q}")
print(f"Output (ViT5 Rewrite) : {pred_clean_q}")
print("=" * 80)